# 06 — Code-DKT (Shi et al., 2022)

Implementação do Code-DKT vanilla conforme Shi et al. (2022), EDM 2022.  
KC = ProblemID, sequências `Run.Program`, extração de paths AST via `javalang`.

**Protocolo de split:** `sequences_bkt_dkt.pkl` (80/20, `random_state=1`, 410 alunos).

---
**Seções neste notebook (Chat 1):**
1. Setup
2. CodeStates
3. Extração de paths — amostra + métricas de transparência
4. Cache de features (paralelizado)
5. Vocabulário A439
6. Tensorização A439 + smoke test forward pass
7. Smoke test de treino (5 épocas)

**Chat 2 continua a partir da Seção 8 (grid search → 10 runs × 5 assignments → Wilcoxon).**

## 1 — Setup

In [ ]:
import os
import sys
import pickle
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Adicionar raiz do projeto ao path
ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.code_features import (
    load_code_states, extract_paths_javalang,
    build_cache, build_vocab, paths_to_tensor, build_code_input_tensor,
)
from src.models.code_dkt import (
    CodeDKTModel, train_code_dkt, predict_code_dkt, train_and_evaluate,
)
from src.evaluation import build_problem_index, compute_auc

print(f"Python {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

import javalang
print(f"javalang {javalang.__version__}")

In [ ]:
# ── Reprodutibilidade (Seção 1.5 do plano) ─────────────────────────────────
SEED = 42

def set_global_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(SEED)

# ── Device ─────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = ROOT / "data" / "CSEDM"
RESULTS_DIR = ROOT / "results"
CACHE_PATH = RESULTS_DIR / "code_features_cache.pkl"

# ── Hiperparâmetros fixos (Shi et al., 2022, Seção 8.1 do plano) ───────────
MAX_PATH_LENGTH = 8
MAX_PATH_WIDTH  = 2
R               = 50   # paths por submissão (Table 3)
MAX_SEQ_LEN     = 50   # comprimento máximo de sequência
M               = 10   # problemas por assignment no CSEDM

DEFAULT_CONFIG = dict(
    hidden_dim  = 128,
    dropout     = 0.0,
    lr          = 0.0005,
    batch_size  = 128,
    epochs      = 40,
    max_len     = MAX_SEQ_LEN,
    R           = R,
)

print("Seed:", SEED)
print("Config default:", DEFAULT_CONFIG)

In [ ]:
# ── Carregar sequências ─────────────────────────────────────────────────────
with open(RESULTS_DIR / "sequences_bkt_dkt.pkl", "rb") as f:
    seqs = pickle.load(f)

ASSIGNMENT_IDS = seqs["assignment_ids"]
print("Assignment IDs:", ASSIGNMENT_IDS)
print("N train A439:", len(seqs["train"][439]))
print("N test  A439:", len(seqs["test"][439]))
print("Colunas events:", seqs["train"][439][0]["events"].columns.tolist())

# Verificar que todos os eventos são Run.Program e CodeStateID é não-nulo
ev_sample = seqs["train"][439][0]["events"]
assert ev_sample["EventType"].unique().tolist() == ["Run.Program"], "Tipo de evento inesperado"
assert ev_sample["CodeStateID"].isnull().sum() == 0, "CodeStateID nulo encontrado"
print("\nVerificações OK: apenas Run.Program, CodeStateID sem nulos.")

## 2 — CodeStates

In [ ]:
print("Carregando CodeStates.csv...")
t0 = time.time()
code_states = load_code_states(DATA_DIR)
print(f"  {len(code_states):,} CodeStateIDs carregados em {time.time()-t0:.1f}s")

# ── Coletar todos os CodeStateIDs únicos das sequências ────────────────────
csids_train: set[str] = set()
csids_test:  set[str] = set()

for aid in ASSIGNMENT_IDS:
    for seq in seqs["train"][aid]:
        csids_train.update(seq["events"]["CodeStateID"].astype(str).values)
    for seq in seqs["test"][aid]:
        csids_test.update(seq["events"]["CodeStateID"].astype(str).values)

csids_all = csids_train | csids_test
print(f"\nCodeStateIDs únicos — train: {len(csids_train):,} | test: {len(csids_test):,} | total: {len(csids_all):,}")

# ── Verificar cobertura no CodeStates.csv ──────────────────────────────────
in_csv = sum(1 for c in csids_all if c in code_states)
print(f"Presentes no CodeStates.csv: {in_csv:,}/{len(csids_all):,} ({100*in_csv/len(csids_all):.1f}%)")

## 3 — Extração de paths — amostra + métricas de transparência

Amostrar 100 submissões `Run.Program` do train A439 para medir taxa de parsing, distribuição de paths e tempo médio (Seção 3.5 do plano).

In [ ]:
# Coletar CodeStateIDs do train A439 (com repetição — um por evento)
csids_a439_train = []
for seq in seqs["train"][439]:
    csids_a439_train.extend(seq["events"]["CodeStateID"].astype(str).values)

rng = random.Random(SEED)
sample_csids = rng.sample(csids_a439_train, min(100, len(csids_a439_train)))
print(f"Amostra: {len(sample_csids)} CodeStateIDs")

# ── Extração com métricas ───────────────────────────────────────────────────
n_success = 0
n_fail    = 0
n_paths_per_sub = []   # paths ANTES da amostragem R=50
times_ms = []
examples = []          # 3 exemplos para inspeção

for csid in sample_csids:
    code = code_states.get(csid, "")
    t0 = time.perf_counter()
    # Extrair SEM limite de R para medir distribuição real
    paths_full = extract_paths_javalang(
        code, MAX_PATH_LENGTH, MAX_PATH_WIDTH, R=99999, seed=SEED
    )
    elapsed_ms = (time.perf_counter() - t0) * 1000
    times_ms.append(elapsed_ms)

    if paths_full:
        n_success += 1
        n_paths_per_sub.append(len(paths_full))
        if len(examples) < 3:
            examples.append((csid, paths_full[:2]))
    else:
        n_fail += 1

total = n_success + n_fail
print(f"\n--- Métricas de transparência (Seção 3.5) ---")
print(f"Parsing javalang: {n_success}/{total} = {100*n_success/total:.1f}% sucesso")
print(f"Uncompilable:     {n_fail}/{total} = {100*n_fail/total:.1f}%")
if n_paths_per_sub:
    print(f"Paths por submissão (submissões com sucesso):")
    print(f"  Mediana: {np.median(n_paths_per_sub):.0f}")
    print(f"  p95:     {np.percentile(n_paths_per_sub, 95):.0f}")
    print(f"  p99:     {np.percentile(n_paths_per_sub, 99):.0f}")
    print(f"  Máx:     {max(n_paths_per_sub)}")
print(f"Tempo médio por submissão: {np.mean(times_ms):.1f} ms")
print(f"Tempo mediano:             {np.median(times_ms):.1f} ms")

In [ ]:
# ── Exemplos de paths extraídos ─────────────────────────────────────────────
print("Exemplos de paths extraídos (start_token, path_str, end_token):")
for csid, path_list in examples:
    print(f"\n  CodeStateID: {csid[:16]}...")
    for start, path_str, end in path_list:
        nodes = path_str.split("@")
        print(f"    ({start}) — {' → '.join(nodes)} — ({end})")

# Estimativa de tempo para cache completo
mean_ms = np.mean(times_ms)
n_cpu = os.cpu_count()
est_seq_min = len(csids_all) * mean_ms / 1000 / 60
est_par_min = est_seq_min / n_cpu
print(f"\nEstimativa para {len(csids_all):,} CodeStateIDs:")
print(f"  Sequencial:          {est_seq_min:.0f} min")
print(f"  Paralelo ({n_cpu} CPUs): {est_par_min:.1f} min")

## 4 — Cache de features (extração paralela)

Extrair paths para todos os CodeStateIDs únicos de train + test de todos os 5 assignments.  
Salvar em `results/code_features_cache.pkl` para evitar re-extração.

In [ ]:
if CACHE_PATH.exists():
    print(f"Cache existente encontrado: {CACHE_PATH}")
    print(f"Tamanho: {CACHE_PATH.stat().st_size / 1e6:.1f} MB")
    with open(CACHE_PATH, "rb") as f:
        cache_raw = pickle.load(f)
    print(f"CodeStateIDs no cache: {len(cache_raw):,}")
else:
    print(f"Extraindo paths para {len(csids_all):,} CodeStateIDs...")
    print(f"Usando multiprocessing.Pool({os.cpu_count()} workers)")
    t0 = time.time()
    cache_raw = build_cache(
        list(csids_all), code_states,
        max_path_length=MAX_PATH_LENGTH,
        max_path_width=MAX_PATH_WIDTH,
        R=R,
        seed=SEED,
        n_workers=None,  # os.cpu_count()
    )
    elapsed = time.time() - t0
    print(f"Extração concluída em {elapsed/60:.1f} min")

    with open(CACHE_PATH, "wb") as f:
        pickle.dump(cache_raw, f, protocol=4)
    print(f"Cache salvo: {CACHE_PATH.stat().st_size / 1e6:.1f} MB")

In [ ]:
# ── Estatísticas do cache completo ─────────────────────────────────────────
n_with_paths = sum(1 for v in cache_raw.values() if v)
n_empty      = sum(1 for v in cache_raw.values() if not v)
paths_counts = [len(v) for v in cache_raw.values() if v]

print(f"Cache completo — {len(cache_raw):,} CodeStateIDs")
print(f"  Com paths:   {n_with_paths:,} ({100*n_with_paths/len(cache_raw):.1f}%)")
print(f"  Sem paths:   {n_empty:,} ({100*n_empty/len(cache_raw):.1f}%) ← 'Uncompilable'")
if paths_counts:
    print(f"  Mediana paths/sub: {np.median(paths_counts):.0f}")
    print(f"  p95:               {np.percentile(paths_counts, 95):.0f}")
    print(f"  (após amostragem R={R})")

## 5 — Vocabulário A439

In [ ]:
# Vocabulário construído APENAS dos CodeStateIDs do train set de A439
# (Seção 4.1 do plano — por assignment para capturar padrões específicos)

csids_a439_train_unique = set()
csids_a439_test_unique  = set()

for seq in seqs["train"][439]:
    csids_a439_train_unique.update(seq["events"]["CodeStateID"].astype(str).values)
for seq in seqs["test"][439]:
    csids_a439_test_unique.update(seq["events"]["CodeStateID"].astype(str).values)

# Cache do train A439
cache_train_a439 = {c: cache_raw[c] for c in csids_a439_train_unique if c in cache_raw}

token_to_idx, path_to_idx = build_vocab(cache_train_a439)

vocab_a439 = dict(
    token_to_idx = token_to_idx,
    path_to_idx  = path_to_idx,
    node_count   = len(token_to_idx),
    path_count   = len(path_to_idx),
)

print(f"Vocabulário A439 (train):")
print(f"  node_count (tokens únicos): {vocab_a439['node_count']:,}")
print(f"  path_count (paths únicos):  {vocab_a439['path_count']:,}")

In [ ]:
# ── % OOV no test set de A439 ───────────────────────────────────────────────
all_test_tokens: list[str] = []
all_test_paths:  list[str] = []

for csid in csids_a439_test_unique:
    for start, path_str, end in cache_raw.get(csid, []):
        all_test_tokens.extend([start, end])
        all_test_paths.append(path_str)

oov_tokens = sum(1 for t in all_test_tokens if t not in token_to_idx)
oov_paths  = sum(1 for p in all_test_paths  if p not in path_to_idx)

print("OOV no test set A439:")
print(f"  Tokens: {oov_tokens:,}/{len(all_test_tokens):,} = {100*oov_tokens/max(1,len(all_test_tokens)):.1f}%")
print(f"  Paths:  {oov_paths:,}/{len(all_test_paths):,}  = {100*oov_paths/max(1,len(all_test_paths)):.1f}%")
print("(OOV → índice 0 = PAD/UNK, esperado para dataset pequeno)")

## 6 — Tensorização A439 + smoke test forward pass

In [ ]:
# ── problem_to_idx A439 ─────────────────────────────────────────────────────
problem_to_idx_a439 = build_problem_index(
    seqs["train"][439] + seqs["test"][439]
)
M_a439 = len(problem_to_idx_a439)
print(f"M (problemas A439): {M_a439}")
print(f"ProblemIDs: {sorted(problem_to_idx_a439.keys())}")

# ── Tensorização ────────────────────────────────────────────────────────────
print("\nConstruindo tensores de treino A439...")
t0 = time.time()

set_global_seed(SEED)
X_train, Y_next_train, mask_train = build_code_input_tensor(
    seqs["train"][439], cache_raw,
    token_to_idx, path_to_idx, problem_to_idx_a439,
    max_len=MAX_SEQ_LEN, R=R,
)
print(f"  Concluído em {time.time()-t0:.1f}s")
print(f"  X_train:      {tuple(X_train.shape)}   ← (N, max_len, 2M + R*3)")
print(f"  Y_next_train: {tuple(Y_next_train.shape)}")
print(f"  mask_train:   {tuple(mask_train.shape)}")

# Verificar dimensões esperadas
expected_last = 2 * M_a439 + R * 3
assert X_train.shape == (len(seqs["train"][439]), MAX_SEQ_LEN, expected_last), \
    f"Shape inesperado: {X_train.shape}"
print(f"\nShape check OK: (N={len(seqs['train'][439])}, L={MAX_SEQ_LEN}, 2M+R*3={expected_last})")
print(f"  2M = {2*M_a439}, R*3 = {R*3}, total = {expected_last}")

In [ ]:
# ── Smoke test forward pass (sem treino) ────────────────────────────────────
set_global_seed(SEED)
model_smoke = CodeDKTModel(
    input_dim   = 2 * M_a439,
    hidden_dim  = DEFAULT_CONFIG["hidden_dim"],
    output_dim  = M_a439,
    node_count  = vocab_a439["node_count"],
    path_count  = vocab_a439["path_count"],
    R           = R,
).to(device)

# Um batch de 4 sequências
x_batch = X_train[:4].to(device)
with torch.no_grad():
    out_smoke = model_smoke(x_batch)

print(f"Forward pass OK")
print(f"  Input:  {tuple(x_batch.shape)}")
print(f"  Output: {tuple(out_smoke.shape)}  ← esperado (4, {MAX_SEQ_LEN}, {M_a439})")
print(f"  Valores saída (min, max): ({out_smoke.min():.4f}, {out_smoke.max():.4f})")
assert out_smoke.shape == (4, MAX_SEQ_LEN, M_a439)
assert 0 <= out_smoke.min() and out_smoke.max() <= 1, "Sigmoid fora de [0,1]"

# Parâmetros do modelo
n_params = sum(p.numel() for p in model_smoke.parameters())
print(f"\nParâmetros totais: {n_params:,}")
del model_smoke
if device.type == "cuda":
    torch.cuda.empty_cache()

## 7 — Smoke test de treino (5 épocas, A439, seed=42)

Verificar que a loss decresce e que `first_auc` > 0.55 (acima de chance).

In [ ]:
smoke_config = {**DEFAULT_CONFIG, "epochs": 5}
print("Smoke test config:", smoke_config)

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

set_global_seed(SEED)
t0 = time.time()
smoke_result = train_and_evaluate(
    seqs["train"][439], seqs["test"][439],
    problem_to_idx_a439, vocab_a439, smoke_config, cache_raw,
    seed=SEED,
)
elapsed_smoke = time.time() - t0

print(f"\n--- Resultado Smoke Test ---")
print(f"all_auc (smoke):    {smoke_result['all_auc']:.4f}")
print(f"first_auc (smoke):  {smoke_result['first_auc']:.4f}")
print(f"Tempo total:        {elapsed_smoke:.1f}s")
print(f"n_train_events:     {smoke_result['n_train_events']:,}")
print(f"n_test_events:      {smoke_result['n_test_events']:,}")

if device.type == "cuda":
    peak_vram = torch.cuda.max_memory_allocated() / 1e6
    print(f"Pico VRAM:          {peak_vram:.0f} MB")

# Critério mínimo para go/no-go
first_auc_smoke = smoke_result["first_auc"]
assert first_auc_smoke > 0.50, f"first_auc smoke abaixo de 0.50: {first_auc_smoke:.4f}"
print(f"\nCritério mínimo (first_auc > 0.50): {'PASSOU' if first_auc_smoke > 0.50 else 'FALHOU'}")

In [ ]:
# ── Verificar convergência da loss ─────────────────────────────────────────
# Re-treinar com prints capturados para verificar que loss[5] < loss[1]
# (o train_and_evaluate já imprimiu as épocas acima; apenas verificação conceitual)
print("Verificação de convergência: observar se loss decresceu entre época 1 e 5.")
print("Se a loss caiu, o modelo está aprendendo e o pipeline está correto.")
print(f"\nfirst_auc smoke = {first_auc_smoke:.4f} > 0.50 → pipeline funcional.")
print("\n✓ Chat 1 concluído com sucesso. Chat 2 continua da Seção 8 (grid search).")

---
## Fronteira Chat 1 → Chat 2

As seções abaixo (8–14) serão implementadas no **Chat 2**.

| Seção | Conteúdo |
|---|---|
| 8 | Grid search (4 configs × A439) — selecionar melhor hiperparâmetros |
| 9 | 10 runs × 5 assignments × 40 épocas |
| 10 | mean ± std all-attempts + first-attempt AUC por assignment |
| 11 | Wilcoxon signed-rank BKT vs DKT vs Code-DKT |
| 12 | Análise qualitativa de paths com maior peso de atenção |
| 13 | Serialização `results/code_dkt_results.pkl` |
| 14 | Sumário comparativo final |

**Antes de iniciar o Chat 2**, verificar:
```bash
.venv/bin/python -c "import javalang; print(javalang.__version__)"
.venv/bin/python -c "import torch; print(torch.cuda.is_available())"
```
E que `results/code_features_cache.pkl` existe e é carregável sem erros.